# Livability Score: Apartment Buildings in Cincinnati

Builds a livability score for apartment buildings in Cincinnati, OH, driven by measurable factors pulled from OpenStreetMap.

This notebook covers:
1. Downloading apartment building footprints from OSM (`osmnx`) and mapping them.
2. Downloading parks from OSM and counting how many fall within 1 km of each building (a simple buffer, not a walking/network distance).
3. Downloading transit stops and schools from OSM and computing each building's straight-line distance to the nearest one of each.
4. Normalizing each factor to a 0-100 scale and combining them into a single `livability_score` using explicit, adjustable weights.
5. Saving the scored buildings to a GeoPackage and GeoJSON.

More factors (crime, amenities, etc.) can be added later by extending `FACTOR_CONFIG` further down - the normalization and combination logic doesn't need to change.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import pandas as pd

PLACE_NAME = "Cincinnati, Ohio, USA"
UTM_CRS = "EPSG:32616"  # UTM zone 16N - the metric CRS for the Cincinnati area

# Notebook lives in scripts/, so data/outputs live one level up
REPO_ROOT = Path.cwd().parent
DATA_DIR = REPO_ROOT / "data" / "livability_score"
OUTPUT_DIR = REPO_ROOT / "outputs" / "livability"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ox.settings.use_cache = True
ox.settings.log_console = False


def clean_for_export(gdf):
    """OSM tag columns can hold list/dict values (e.g. relation member
    lists) that the GeoJSON/GPKG drivers can't serialize - stringify any
    column like that before writing to disk."""
    cleaned = gdf.copy()
    for col in cleaned.columns:
        if col == "geometry":
            continue
        if cleaned[col].apply(lambda v: isinstance(v, (list, dict))).any():
            cleaned[col] = cleaned[col].astype(str)
    return cleaned

## Download apartment buildings

In [ ]:
apartments_raw = ox.features_from_place(PLACE_NAME, tags={"building": "apartments"})

# Most building=apartments tags are on ways/relations (polygons); a few are
# on individual nodes (points) and don't represent a footprint, so drop them
apartments = apartments_raw[apartments_raw.geom_type.isin(["Polygon", "MultiPolygon"])].reset_index()

print(f"Apartment building polygons found in Cincinnati: {len(apartments)}")

## Interactive map

In [ ]:
tooltip_cols = [c for c in ["name", "addr:housenumber", "addr:street"] if c in apartments.columns]

m = apartments.explore(
    color="steelblue",
    style_kwds={"fillOpacity": 0.6},
    tooltip=tooltip_cols or None,
    name="Apartment Buildings",
)
m

In [ ]:
apartments_path = DATA_DIR / "apartment_buildings.geojson"
clean_for_export(apartments).to_file(apartments_path, driver="GeoJSON")
print(f"Saved {len(apartments)} apartment buildings to {apartments_path.relative_to(REPO_ROOT)}")

## Download parks

In [ ]:
parks_raw = ox.features_from_place(PLACE_NAME, tags={"leisure": "park"})
parks = parks_raw.reset_index()

print(f"Parks found in Cincinnati: {len(parks)}")

parks_path = DATA_DIR / "parks.geojson"
clean_for_export(parks).to_file(parks_path, driver="GeoJSON")
print(f"Saved {len(parks)} parks to {parks_path.relative_to(REPO_ROOT)}")

## Parks within 1 km of each apartment building

A simple buffer: expand each building's footprint by 1 km and count how many parks that expanded shape touches. This is not a walking or road-network distance - just straight-line proximity.

In [ ]:
# OSM data is in degrees (EPSG:4326) - reproject to a metric CRS before
# buffering, otherwise "1000" wouldn't mean 1000 meters
apartments_utm = apartments.to_crs(UTM_CRS)
parks_utm = parks.to_crs(UTM_CRS)

buffered = apartments_utm[["geometry"]].copy()
buffered["geometry"] = apartments_utm.geometry.buffer(1000)

# how="left" keeps every building even when it matches zero parks
# (index_right is NaN for those rows)
joined = gpd.sjoin(buffered, parks_utm[["geometry"]], predicate="intersects", how="left")
apartments["parks_within_1km"] = joined.groupby(joined.index)["index_right"].apply(lambda s: s.notna().sum())

apartments["parks_within_1km"].describe()

### Distribution: parks within 1 km

In [ ]:
max_count = apartments["parks_within_1km"].max()
ax = apartments["parks_within_1km"].plot(kind="hist", bins=range(0, max_count + 2), edgecolor="black")
ax.set_xlabel("Parks within 1 km")
ax.set_ylabel("Number of apartment buildings")
ax.set_title("Distribution of parks within 1 km of each apartment building")
plt.show()

### Sanity check: `parks_within_1km`

Summary statistics, plus the 10 buildings with the most parks nearby and the 10 with zero, so specific locations can be checked against local knowledge of Cincinnati rather than just the aggregate shape above.

In [ ]:
def build_identifier(row):
    """Best available human-readable label for a building - OSM tagging is
    inconsistent, so fall back from name -> street address -> OSM id."""
    name = row.get("name")
    if pd.notna(name) and str(name).strip():
        return str(name)
    housenumber, street = row.get("addr:housenumber"), row.get("addr:street")
    if pd.notna(housenumber) and pd.notna(street):
        return f"{housenumber} {street}"
    return f"Unnamed ({row.get('element')} {row.get('id')})"


apartments["identifier"] = apartments.apply(build_identifier, axis=1)

print("parks_within_1km summary statistics:")
apartments["parks_within_1km"].describe()

In [ ]:
print("Top 10 apartment buildings by parks within 1 km:")
top10_parks = apartments.nlargest(10, "parks_within_1km")[["identifier", "parks_within_1km"]].reset_index(drop=True)
top10_parks

In [ ]:
zero_park_buildings = apartments[apartments["parks_within_1km"] == 0]
print(f"Buildings with zero parks within 1 km: {len(zero_park_buildings)}")

bottom10_parks = zero_park_buildings[["identifier", "parks_within_1km"]].sample(
    min(10, len(zero_park_buildings)), random_state=0
).reset_index(drop=True)
bottom10_parks

## Download transit stops

Bus stops (`highway=bus_stop`) and rail/tram stops (`railway=stop`, `railway=station`, `railway=tram_stop`).

In [ ]:
stops_raw = ox.features_from_place(
    PLACE_NAME,
    tags={"highway": "bus_stop", "railway": ["stop", "station", "tram_stop"]},
)
stops = stops_raw.reset_index()

print(f"Transit stops found in Cincinnati: {len(stops)}")

stops_path = DATA_DIR / "transit_stops.geojson"
clean_for_export(stops).to_file(stops_path, driver="GeoJSON")
print(f"Saved {len(stops)} transit stops to {stops_path.relative_to(REPO_ROOT)}")

## Distance to nearest transit stop

Straight-line distance in meters from each apartment building to the nearest transit stop, computed in the same projected CRS (UTM 16N) used for the parks buffer - not a walking or road-network distance.

In [ ]:
stops_utm = stops.to_crs(UTM_CRS)

# sjoin_nearest is GeoPandas' R-tree-backed nearest-neighbor join - the right
# tool for "nearest", as opposed to the buffer+count approach used for parks
nearest = gpd.sjoin_nearest(
    apartments_utm[["geometry"]], stops_utm[["geometry"]], distance_col="distance_to_transit_m"
)
# A tied nearest distance can produce more than one match per building - keep the smallest
apartments["distance_to_transit_m"] = nearest.groupby(nearest.index)["distance_to_transit_m"].min()

apartments["distance_to_transit_m"].describe()

### Distribution: distance to nearest transit stop

In [ ]:
ax = apartments["distance_to_transit_m"].plot(kind="hist", bins=30, edgecolor="black")
ax.set_xlabel("Distance to nearest transit stop (m)")
ax.set_ylabel("Number of apartment buildings")
ax.set_title("Distribution of distance to nearest transit stop")
plt.show()

### Sanity check: `distance_to_transit_m`

The 10 buildings closest to a transit stop and the 10 farthest, reusing the `identifier` column built above.

In [ ]:
print("10 apartment buildings nearest to a transit stop:")
nearest10_transit = apartments.nsmallest(10, "distance_to_transit_m")[["identifier", "distance_to_transit_m"]].reset_index(drop=True)
nearest10_transit

In [ ]:
print("10 apartment buildings farthest from a transit stop:")
farthest10_transit = apartments.nlargest(10, "distance_to_transit_m")[["identifier", "distance_to_transit_m"]].reset_index(drop=True)
farthest10_transit

## Download schools

In [ ]:
schools_raw = ox.features_from_place(PLACE_NAME, tags={"amenity": "school"})
schools = schools_raw.reset_index()

print(f"Schools found in Cincinnati: {len(schools)}")

schools_path = DATA_DIR / "schools.geojson"
clean_for_export(schools).to_file(schools_path, driver="GeoJSON")
print(f"Saved {len(schools)} schools to {schools_path.relative_to(REPO_ROOT)}")

## Distance to nearest school

Straight-line distance in meters from each apartment building to the nearest school, computed the same way as the transit-stop distance above (UTM 16N, not a walking or road-network distance). Schools are tagged as either points or building polygons in OSM; `sjoin_nearest` handles both.

In [ ]:
schools_utm = schools.to_crs(UTM_CRS)

nearest_school = gpd.sjoin_nearest(
    apartments_utm[["geometry"]], schools_utm[["geometry"]], distance_col="distance_to_school_m"
)
# A tied nearest distance can produce more than one match per building - keep the smallest
apartments["distance_to_school_m"] = nearest_school.groupby(nearest_school.index)["distance_to_school_m"].min()

apartments["distance_to_school_m"].describe()

### Distribution: distance to nearest school

In [ ]:
ax = apartments["distance_to_school_m"].plot(kind="hist", bins=30, edgecolor="black")
ax.set_xlabel("Distance to nearest school (m)")
ax.set_ylabel("Number of apartment buildings")
ax.set_title("Distribution of distance to nearest school")
plt.show()

### Sanity check: `distance_to_school_m`

The 10 buildings closest to a school and the 10 farthest, reusing the `identifier` column built above.

In [ ]:
print("10 apartment buildings nearest to a school:")
nearest10_school = apartments.nsmallest(10, "distance_to_school_m")[["identifier", "distance_to_school_m"]].reset_index(drop=True)
nearest10_school

In [ ]:
print("10 apartment buildings farthest from a school:")
farthest10_school = apartments.nlargest(10, "distance_to_school_m")[["identifier", "distance_to_school_m"]].reset_index(drop=True)
farthest10_school

## Livability score

Each factor is normalized to a 0-100 scale (100 = best) and combined into a single `livability_score` as a weighted average. Two things are stated explicitly per factor, both easy to see and adjust in `FACTOR_CONFIG` below:

- **Direction**: whether a *higher* raw value is better (e.g. more parks nearby) or a *lower* raw value is better (e.g. distance to the nearest transit stop or school, where being closer is better).
- **Weight**: how much that factor counts toward the combined score, relative to the others. Transit and school access are weighted equally (0.35 each) and given slightly more weight than parks (0.30), reflecting a judgment call that daily-access factors matter a bit more than recreational proximity - adjust these to reflect your own priorities.

`parks_within_1km`, `distance_to_transit_m`, and `distance_to_school_m` are combined below, but the framework generalizes to more factors - adding one is just adding an entry to `FACTOR_CONFIG` and a column with that factor's raw values.

In [ ]:
def normalize_factor(series, direction):
    """Min-max scales a column to 0-100, inverting it first when direction
    is "lower_is_better" - so a higher normalized value always means
    "better", regardless of which way the raw factor points."""
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series(100.0, index=series.index)
    if direction == "higher_is_better":
        return (series - lo) / (hi - lo) * 100
    elif direction == "lower_is_better":
        return (hi - series) / (hi - lo) * 100
    else:
        raise ValueError(f"Unknown direction: {direction!r}")


# Explicit, unequal weights - transit and school access weighted more
# heavily than parks. Each factor still just needs a raw column, a
# direction, and a weight.
FACTOR_CONFIG = {
    "distance_to_transit_m": {"direction": "lower_is_better", "weight": 0.35},
    "distance_to_school_m": {"direction": "lower_is_better", "weight": 0.35},
    "parks_within_1km": {"direction": "higher_is_better", "weight": 0.30},
}

total_weight = sum(cfg["weight"] for cfg in FACTOR_CONFIG.values())
assert abs(total_weight - 1.0) < 1e-9, f"FACTOR_CONFIG weights must sum to 1.0, got {total_weight}"

for factor, cfg in FACTOR_CONFIG.items():
    apartments[f"{factor}_norm"] = normalize_factor(apartments[factor], cfg["direction"])

apartments["livability_score"] = sum(
    apartments[f"{factor}_norm"] * cfg["weight"] for factor, cfg in FACTOR_CONFIG.items()
) / total_weight

print(f"Weights sum to {total_weight:.2f}. Final FACTOR_CONFIG:")
pd.DataFrame([
    {"factor": factor, "direction": cfg["direction"], "weight": cfg["weight"]}
    for factor, cfg in FACTOR_CONFIG.items()
])

### Sensitivity check: chosen weights vs. equal weights

Computes `livability_score_equal` - the same three normalized factors averaged with equal weights - purely as a comparison column. It does not replace `livability_score`. The Spearman rank correlation between the two shows how much the 0.35/0.35/0.30 weighting actually reshuffles the ranking versus just averaging the factors equally.

In [ ]:
apartments["livability_score_equal"] = sum(
    apartments[f"{factor}_norm"] for factor in FACTOR_CONFIG
) / len(FACTOR_CONFIG)

spearman_corr = apartments["livability_score"].corr(apartments["livability_score_equal"], method="spearman")
print(f"Spearman rank correlation (chosen weights vs. equal weights): {spearman_corr:.4f}")

### Distribution: livability score

In [ ]:
ax = apartments["livability_score"].plot(kind="hist", bins=20, edgecolor="black")
ax.set_xlabel("Livability score (0-100)")
ax.set_ylabel("Number of apartment buildings")
ax.set_title("Distribution of livability scores")
plt.show()

print("livability_score summary statistics:")
apartments["livability_score"].describe()

## Save scored buildings

In [ ]:
scored_path = OUTPUT_DIR / "apartment_buildings_scored.gpkg"
clean_for_export(apartments).to_file(scored_path, driver="GPKG")
print(f"Saved {len(apartments)} scored apartment buildings to {scored_path.relative_to(REPO_ROOT)}")

In [ ]:
scored_geojson_path = OUTPUT_DIR / "apartment_buildings_scored.geojson"
# apartments is never reprojected in place (only the apartments_utm copy used
# for buffering is) - it's already EPSG:4326, same as the GeoPackage export
clean_for_export(apartments).to_file(scored_geojson_path, driver="GeoJSON")
print(f"Saved {len(apartments)} scored apartment buildings to {scored_geojson_path.relative_to(REPO_ROOT)}")

In [ ]:
# Re-read both files back to confirm the writes actually succeeded, and
# report what they contain
for path in [scored_path, scored_geojson_path]:
    written = gpd.read_file(path)
    print(f"{path.name}: {len(written)} rows, {len(written.columns)} columns")
    print(list(written.columns))
    print()